In [ ]:
"""
Build Taxonomy Pipeline - Multi-View Embedding Approach

This notebook implements a multi-view embedding strategy for taxonomy preparation:
- Separate embeddings for label, definition, and examples (no concatenation)
- Incremental learning placeholders for future training integration
- Avoids prototype dilution for short-query matching
"""
import pandas as pd
import numpy as np
from typing import Dict, List, Optional

import taxomind.utils.taxonomy_utils as taxo_utils

In [ ]:
# Configuration
TAXONOMY_KEY = "ISIC"  # Can be changed to "ISCO" or other taxonomy keys

In [ ]:
# Load Kedro context
%load_ext kedro.ipython
%reload_kedro

In [ ]:
# Step 0.1: Load taxonomy from catalog (PartitionedDataset)
from taxomind.utils.taxonomy_utils import get_partition_by_key

print(f"Loading taxonomy: {TAXONOMY_KEY}")

# Load partitioned dataset (returns dict of callables)
taxonomy_partitions = catalog.load('taxonomy_definition')

# Get the specific partition using utility function
taxonomy_df = get_partition_by_key(taxonomy_partitions, TAXONOMY_KEY)

print(f"Loaded {len(taxonomy_df)} nodes")
print(f"Columns: {list(taxonomy_df.columns)}")
print(f"\nFirst few rows:")
taxonomy_df.head()

In [ ]:
# Step 0.1: Build Node Prototype Fields (Multi-View)
# Extract three separate text views per node (no concatenation)

def normalise_prototype_views(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    df['code'] = df['code'].apply(taxo_utils.normalize_code)
    df['parentCode'] = df['parentCode'].apply(taxo_utils.normalize_parent)
    df['label'] = df['label'].apply(taxo_utils.normalize_text)
    df['examples'] = df['examples'].apply(taxo_utils.normalize_text)
    df['definition'] = df['definition'].apply(taxo_utils.normalize_text)    
    return df

taxonomy_with_views = normalise_prototype_views(taxonomy_df)

# Show statistics
print(f"\nPrototype View Statistics:")


In [ ]:
# Step 0.2: Embed Taxonomy Prototypes (Zero-Shot, Multi-View)
# Compute separate normalized embeddings for each view

from taxomind.utils import embedding_utils

def embed_prototype_views(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """
    Create separate embeddings for label, definition, and examples views.
    
    Computes:
    - E_label[n]: Always present (from label)
    - E_def[n]: Only if definition is not None
    - E_ex[n]: Only if examples is not None
    
    All embeddings are normalized.
    
    Returns DataFrame with added columns: E_label, E_def, E_ex, embedding_model_name
    """
    df = df.copy()
    
    print(f"Embedding with model: {model_name}")
    embedding_model = embedding_utils.load_embedding_model(model_name)
    
    # Embed label view (always present)
    print("  Embedding labels...")
    label_texts = df['label'].fillna('').tolist()
    label_embeddings, _ = embedding_utils.encode_texts(
        embedding_model,
        label_texts,
        embed_all=True,
        batch_size=32,
        show_progress_bar=True,
    )
    # Use list comprehension to properly assign each embedding
    df['E_label'] = [emb for emb in label_embeddings]
    
    # Embed definition view (only where present)
    print("  Embedding definitions...")
    def_mask = df['definition'].notna()
    # Initialize with None for all rows
    df['E_def'] = [None] * len(df)
    
    if def_mask.any():
        def_texts = df.loc[def_mask, 'definition'].tolist()
        def_embeddings, _ = embedding_utils.encode_texts(
            embedding_model,
            def_texts,
            embed_all=True,
            batch_size=32,
            show_progress_bar=True,
        )
        # Assign embeddings to the masked rows
        def_indices = df.index[def_mask].tolist()
        for idx, emb in zip(def_indices, def_embeddings):
            df.at[idx, 'E_def'] = emb
    
    # Embed examples view (only where present)
    print("  Embedding examples...")
    ex_mask = df['examples'].notna()
    # Initialize with None for all rows
    df['E_ex'] = [None] * len(df)
    
    if ex_mask.any():
        ex_texts = df.loc[ex_mask, 'examples'].tolist()
        ex_embeddings, _ = embedding_utils.encode_texts(
            embedding_model,
            ex_texts,
            embed_all=True,
            batch_size=32,
            show_progress_bar=True,
        )
        # Assign embeddings to the masked rows
        ex_indices = df.index[ex_mask].tolist()
        for idx, emb in zip(ex_indices, ex_embeddings):
            df.at[idx, 'E_ex'] = emb
    
    # Store model name
    df['embedding_model_name'] = model_name
    
    print(f"  ✓ Embedded {len(label_embeddings)} labels")
    print(f"  ✓ Embedded {def_mask.sum()} definitions")
    print(f"  ✓ Embedded {ex_mask.sum()} examples")
    
    return df

# Get embedding model from parameters
embedding_model = context.params['model_name']
taxonomy_embedded = embed_prototype_views(taxonomy_with_views, embedding_model)

# Verify embeddings
print(f"
Embedding columns added:")
print(f"  E_label present: {taxonomy_embedded['E_label'].notna().sum()}")
print(f"  E_def present: {taxonomy_embedded['E_def'].notna().sum()}")
print(f"  E_ex present: {taxonomy_embedded['E_ex'].notna().sum()}")


In [ ]:
# Step 0.3: Initialize Incremental Storage (Per Node)
# Add placeholders for future incremental learning from training data

def initialize_incremental_storage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Initialize incremental learning storage for each taxonomy node.
    
    Adds:
    - C_emb_node: Centroid embedding from training data (None initially)
    - k_emb_node: Count of training samples (0 initially)
    
    These will be updated during incremental training and used for 
    computing effective evidence embeddings at inference time:
    
    β_n = k_emb_node[n] / (k_emb_node[n] + τ)
    C_emb_node_eff[n] = normalize((1-β_n)·E_label[n] + β_n·C_emb_node[n])
    
    Returns DataFrame with added columns: C_emb_node, k_emb_node
    """
    df = df.copy()
    
    # Initialize centroid embedding (None = no training data yet)
    # Use list to avoid pandas broadcasting issues
    df['C_emb_node'] = [None] * len(df)
    
    # Initialize sample count (0 = no training data yet)
    df['k_emb_node'] = 0
    
    print(f"Initialized incremental storage for {len(df)} nodes")
    print(f"  C_emb_node: All None (no training data)")
    print(f"  k_emb_node: All 0 (no training samples)")
    
    return df

taxonomy_final = initialize_incremental_storage(taxonomy_embedded)

# Show final structure
print(f"\nFinal taxonomy structure:")
print(f"  Total nodes: {len(taxonomy_final)}")
print(f"  Columns: {list(taxonomy_final.columns)}")
print(f"\nKey columns:")
print(f"  - E_label: {taxonomy_final['E_label'].notna().sum()} embeddings")
print(f"  - E_def: {taxonomy_final['E_def'].notna().sum()} embeddings")
print(f"  - E_ex: {taxonomy_final['E_ex'].notna().sum()} embeddings")
print(f"  - C_emb_node: {taxonomy_final['C_emb_node'].notna().sum()} (should be 0)")
print(f"  - k_emb_node: sum={taxonomy_final['k_emb_node'].sum()} (should be 0)")

taxonomy_final.head(3)

In [ ]:
# Save to catalog (taxonomy_embedded PartitionedDataset)
print(f"\nSaving taxonomy to catalog...")
print(f"  Dataset: taxonomy_embedded")
print(f"  Partition key: {TAXONOMY_KEY}")

# Create dictionary with taxonomy key as partition
taxonomy_to_save = {TAXONOMY_KEY: taxonomy_final}

# Save to catalog
catalog.save('taxonomy_embedded', taxonomy_to_save)

print(f"✓ Taxonomy saved successfully!")
print(f"  Location: data/03_primary/taxonomies/embedded/{TAXONOMY_KEY}.parquet")